> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 图存储模块（Graph Store）

## 概览

图存储模块为属性图数据库提供统一接口。它支持多个后端（Neo4j、FalkorDB），提供存储、查询和分析图数据的完整功能。

### 核心特性

- **多后端支持**：Neo4j（企业级）、FalkorDB（基于 Redis）
- **完整 CRUD 操作**：节点和关系的创建、读取、更新、删除
- **Cypher 查询语言**：以 OpenCypher 支持执行复杂图查询
- **图分析**：内置中心性、社区发现、路径查找算法
- **批量操作**：带进度追踪的优化批量加载
- **事务支持**：支持回滚的 ACID 事务
- **索引管理**：创建和管理索引以优化性能
- **便捷函数**：面向常见操作的简单函数式 API

### 先看主线

本篇性质变了：前面 07、08 篇把图建在内存里，notebook 一关图就没了。这一篇把图交给专业图数据库——连上后端、存进去、查出来、养起来。十二步分三段：

**连上与写入（第 1–3 步）—— 建库**

1. 第 1 步 连接：初始化 `GraphStore`，连上后端
2. 第 2 步 建节点：实体入库
3. 第 3 步 连关系：边入库

**查询与分析（第 4–6 步）—— 用库**

4. 第 4 步 查询：按标签、属性、ID 找节点和关系
5. 第 5 步 Cypher：用图查询语言表达复杂模式
6. 第 6 步 图分析：邻居遍历、最短路径

**维护与收尾（第 7–12 步）—— 养库**

7. 第 7–8 步 改与删：更新属性、删除节点和关系
8. 第 9–11 步 管与提速：统计体检、便捷函数、索引
9. 第 12 步 关闭：释放连接

两点提醒：其一，本篇与前面的 notebook 不同，需要一个运行中的 Neo4j 或 FalkorDB 服务器才能真正跑通；其二，第 5 步的参数化查询和第 8 步的 detach 删除，是两个最值得记住的实践点。

### 学习目标

完成本 notebook 后，你将能够：

1. 用不同后端初始化和配置 GraphStore
2. 对节点和关系执行 CRUD 操作
3. 执行 Cypher 查询完成复杂图操作
4. 使用图分析算法（最短路径、邻居、中心性）
5. 更新和删除图数据
6. 使用批量操作高效加载数据
7. 使用便捷函数和配置管理
8. 为你的场景选择合适的后端

---

## 安装

### 核心安装

```bash
# 安装 Semantica
pip install semantica

# 或安装全部可选依赖
pip install semantica[all]
```

### 后端专属依赖

```bash
# Neo4j（需要 Neo4j 服务器）
pip install neo4j

# FalkorDB（需要 Redis/FalkorDB 服务器）
pip install falkordb
```

### Docker 部署（可选）

FalkorDB 可以用 Docker 跑：

```bash
docker run -p 6379:6379 -p 3000:3000 -it --rm \
  -v ./data:/var/lib/falkordb/data \
  falkordb/falkordb
```

---

## 后端对比

| 后端 | 最适合 | 部署方式 | 特性 |
|---------|----------|------------|----------|
| **Neo4j** | 企业应用、生产系统 | 服务器/云 | 完整 Cypher、APOC 过程、多数据库 |
| **FalkorDB** | LLM 应用、实时系统、高性能 | 基于 Redis | 极快、稀疏矩阵运算 |

**建议**：企业生产系统选 **Neo4j**，高性能实时应用选 **FalkorDB**。

In [ ]:
!pip install semantica


## 第 1 步：初始化图存储

主线从建库开始：用你选定的后端初始化一个 `GraphStore` 实例。本教程用 **Neo4j**（需要一个运行中的服务器）。

In [ ]:
from semantica.graph_store import GraphStore

# Neo4j AuraDB Connection Details
# Replace these values with your actual AuraDB credentials
store = GraphStore(
    backend="neo4j",
    uri="Your URI",  # Your AuraDB Instance URI
    user="neo4j",
    password="Your Password"  # Please enter your password here
)

# Connect to the database
store.connect()
print("Connected to graph database successfully!")

## 第 2 步：节点操作

### 创建节点

节点代表图里的实体。每个节点可以有：
- **标签（Labels）**：类别/类型（如 `Person`、`Company`、`Location`）
- **属性（Properties）**：键值对（如 `{"name": "Alice", "age": 30}`）

In [ ]:
# Create individual nodes with labels and properties
apple = store.create_node(
    labels=["Company"],
    properties={"name": "Apple Inc.", "founded": 1976, "industry": "Technology"}
)
print(f"Created company node: {apple.get('properties', {}).get('name')} (ID: {apple.get('id')})")

tim_cook = store.create_node(
    labels=["Person"],
    properties={"name": "Tim Cook", "title": "CEO", "age": 63}
)
print(f"Created person node: {tim_cook.get('properties', {}).get('name')} (ID: {tim_cook.get('id')})")

cupertino = store.create_node(
    labels=["Location"],
    properties={"name": "Cupertino", "state": "California", "country": "USA"}
)
print(f"Created location node: {cupertino.get('properties', {}).get('name')} (ID: {cupertino.get('id')})")


In [ ]:
# Create multiple nodes in batch (more efficient for large datasets)
other_companies = store.create_nodes([
    {"labels": ["Company"], "properties": {"name": "Microsoft", "founded": 1975}},
    {"labels": ["Company"], "properties": {"name": "Google", "founded": 1998}},
    {"labels": ["Company"], "properties": {"name": "Amazon", "founded": 1994}},
])
print(f"Created {len(other_companies)} company nodes in batch")


## 第 3 步：关系操作

### 创建关系

关系连接节点，代表实体之间的连接。每条关系有：
- **类型（Type）**：关系类型（如 `CEO_OF`、`LOCATED_IN`、`KNOWS`）
- **属性（Properties）**：键值对（如 `{"since": 2011}`）
- **方向（Direction）**：从 `start_node_id` 指向 `end_node_id`

节点和关系入库，图就装进了数据库。接下来换段：把数据查出来用。

In [ ]:
# Create relationships between nodes
ceo_rel = store.create_relationship(
    start_node_id=tim_cook["id"],
    end_node_id=apple["id"],
    rel_type="CEO_OF",
    properties={"since": 2011}
)
print(f"Created relationship: {ceo_rel.get('type')} (ID: {ceo_rel.get('id')})")

location_rel = store.create_relationship(
    start_node_id=apple["id"],
    end_node_id=cupertino["id"],
    rel_type="HEADQUARTERED_IN",
    properties={"since": 1977}
)
print(f"Created relationship: {location_rel.get('type')} (ID: {location_rel.get('id')})")


## 第 4 步：查询节点和关系

### 读取节点

可以按标签、属性或节点 ID 查询节点。

In [ ]:
# Get nodes by label
companies = store.get_nodes(labels=["Company"], limit=10)
print(f"Found {len(companies)} companies:")
for company in companies:
    name = company.get('properties', {}).get('name', 'Unknown')
    founded = company.get('properties', {}).get('founded', 'N/A')
    print(f"  - {name} (founded: {founded})")

# Get a specific node by ID
if apple.get('id'):
    node = store.get_node(node_id=apple["id"])
    print(f"\nRetrieved node by ID: {node.get('properties', {}).get('name')}")


In [ ]:
# Get relationships for a node
relationships = store.get_relationships(node_id=apple["id"], direction="both")
print(f"Found {len(relationships)} relationships for Apple:")
for rel in relationships:
    rel_type = rel.get('type', 'Unknown')
    props = rel.get('properties', {})
    print(f"  - {rel_type}: {props}")

# Get relationships by type and direction
if tim_cook.get('id'):
    outgoing = store.get_relationships(
        node_id=tim_cook["id"],
        rel_type="CEO_OF",
        direction="out"
    )
    print(f"\nOutgoing CEO_OF relationships: {len(outgoing)}")


## 第 5 步：执行 Cypher 查询

Cypher 是强大的图查询语言，能表达复杂的图模式和操作。图存储模块在所有后端上都支持 **OpenCypher** 语法。留意下面第二个 cell 的参数化查询——既防注入又利于缓存，是生产环境的推荐写法。

In [ ]:
# Execute a Cypher query to find CEO relationships
results = store.execute_query("""
    MATCH (p:Person)-[r:CEO_OF]->(c:Company)
    RETURN p.name as person, c.name as company, r.since as since
""")

print("CEO Relationships:")
for record in results.get("records", []):
    person = record.get('person', 'Unknown')
    company = record.get('company', 'Unknown')
    since = record.get('since', 'N/A')
    print(f"  - {person} is CEO of {company} since {since}")


In [ ]:
# Parameterized query (safer and more efficient)
results = store.execute_query(
    "MATCH (c:Company) WHERE c.founded > $year RETURN c.name, c.founded ORDER BY c.founded",
    parameters={"year": 1990}
)

print("Companies founded after 1990:")
for record in results.get("records", []):
    name = record.get('c.name', 'Unknown')
    founded = record.get('c.founded', 'N/A')
    print(f"  - {name} (founded: {founded})")


## 第 6 步：图分析

图存储模块提供若干内置图分析算法，用于分析图结构。查询到此告一段落——从"按条件找"升级到"按结构算"。

In [ ]:
# Get neighbors of a node (traverse the graph)
if apple.get('id'):
    neighbors = store.get_neighbors(
        node_id=apple["id"],
        direction="both",
        depth=2
    )
    
    print(f"Found {len(neighbors)} neighbors (up to depth 2) for Apple:")
    for neighbor in neighbors:
        name = neighbor.get('properties', {}).get('name', 'Unknown')
        labels = neighbor.get('labels', [])
        print(f"  - {name} ({', '.join(labels)})")


In [ ]:
# Find shortest path between two nodes
if tim_cook.get('id') and cupertino.get('id'):
    path = store.shortest_path(
        start_node_id=tim_cook["id"],
        end_node_id=cupertino["id"],
        max_depth=5
    )
    
    if path:
        print(f"Shortest path found:")
        print(f"  - Path length: {path.get('length')}")
        print(f"  - Nodes in path: {len(path.get('nodes', []))}")
        print(f"  - Relationships: {len(path.get('relationships', []))}")
    else:
        print("No path found between the nodes")


## 第 7 步：更新操作

数据入库不是一锤子买卖。可以用 `update_node` 方法更新节点属性。

In [ ]:
# Update node properties (merge mode - adds/updates properties)
if tim_cook.get('id'):
    updated = store.update_node(
        node_id=tim_cook["id"],
        properties={"age": 64, "title": "CEO & President"},
        merge=True  # Merge with existing properties
    )
    print(f"Updated node: {updated.get('properties', {}).get('name')}")
    print(f"  New age: {updated.get('properties', {}).get('age')}")
    print(f"  New title: {updated.get('properties', {}).get('title')}")

# Example: Replace all properties (merge=False)
# updated = store.update_node(
#     node_id=node_id,
#     properties={"name": "New Name"},
#     merge=False  # Replace all properties
# )


## 第 8 步：删除操作

需要时可以删除节点和关系。留意 detach 的语义：`detach=True` 连带删除该节点的所有关系，`detach=False` 只删节点本身（有关系挂在上面时会失败）。

In [ ]:
# Delete a relationship
if location_rel.get('id'):
    deleted = store.delete_relationship(rel_id=location_rel["id"])
    if deleted:
        print(f"Deleted relationship (ID: {location_rel['id']})")

# Delete a node (with detach=True to also delete its relationships)
# WARNING: This will delete the node and all its relationships
# Uncomment to test:
# if cupertino.get('id'):
#     deleted = store.delete_node(node_id=cupertino["id"], detach=True)
#     if deleted:
#         print(f"Deleted node: {cupertino.get('properties', {}).get('name')}")

print("\nTip: Use detach=True to delete a node and all its relationships")
print("     Use detach=False to only delete the node (fails if relationships exist)")


## 第 9 步：图统计

给图做个体检：获取图谱的全面统计信息。

In [ ]:
# Get comprehensive graph statistics
stats = store.get_stats()

print("Graph Statistics:")
print(f"  Total nodes: {stats.get('node_count', 'N/A')}")
print(f"  Total relationships: {stats.get('relationship_count', 'N/A')}")
print(f"\nNode labels:")
for label, count in stats.get('label_counts', {}).items():
    print(f"  - {label}: {count} nodes")
print(f"\nRelationship types:")
for rel_type, count in stats.get('relationship_type_counts', {}).items():
    print(f"  - {rel_type}: {count} relationships")


## 第 10 步：便捷函数

不想实例化类？图存储模块还提供函数式 API，把常见操作包装成简单函数。

In [ ]:
# Using convenience functions (alternative to class methods)
from semantica.graph_store import (
    create_node,
    create_relationship,
    get_nodes,
    execute_query,
    shortest_path
)

# These functions work with a default store instance
# For this example, we'll continue using the store instance we created

# Example: Using convenience functions
# node = create_node(
#     labels=["Person"],
#     properties={"name": "Alice", "age": 30}
# )

print("Convenience functions available:")
print("  - create_node, create_nodes")
print("  - create_relationship, create_relationships")
print("  - get_nodes, get_relationships")
print("  - update_node, delete_node")
print("  - execute_query, shortest_path, get_neighbors")
print("  - run_analytics")


## 第 11 步：索引管理

图大了查询会慢。创建索引可以提升查询性能，对大图尤其明显。

In [ ]:
# Create an index on a node property for faster lookups
# This is especially useful for frequently queried properties

index_created = store.create_index(
    label="Company",
    property_name="name",
    index_type="btree"  # Default index type
)

if index_created:
    print("Created index on Company.name for faster queries")
else:
    print("Index may already exist or not be supported by this backend")

# Note: Index creation support varies by backend
# Neo4j: Full support for various index types
# FalkorDB: Limited index support


## 第 12 步：清理

用完记得关闭连接，释放资源。

In [ ]:
# Close the connection
store.close()
print("Connection closed successfully")


## 小结

本 notebook 介绍了图存储模块——支持 Neo4j 和 FalkorDB 的属性图数据库统一接口。

### 回顾主线

十二步三段：**建库**（连接、节点、关系入库）让图有了持久的家；**用库**（查询、Cypher、图分析）让数据能被找出来、算起来；**养库**（更新删除、统计、索引、关闭）让库长期健康。CRUD 加索引，就是一个图数据库的完整生命周期。

### 你学到了什么

- **CRUD 操作**：节点和关系的创建、读取、更新、删除
- **Cypher 查询**：用 OpenCypher 语法执行复杂图查询
- **图分析**：最短路径、邻居遍历和中心性算法
- **批量操作**：面向大数据集的高效批量加载
- **索引管理**：通过索引优化性能

### 核心要点

- **后端选择**：生产环境用 Neo4j，高性能应用用 FalkorDB
- **最佳实践**：使用批量操作、参数化查询，并正确管理连接
- **下一步**：探索高级分析、图谱质量和可视化模块